# FIEP — Genossensor de RNA: o voltamograma diz se há contaminação?

Notebook do fluxo do projeto, do `.nox` ao modelo. Cada eletrodo impresso (SPCE) recebe um **elemento
de captura** (oligonucleotídeo) e é medido em DPV: sem o alvo é o **branco** (não contaminado); com o
**RNA da bactéria** é o **alvo** (contaminado). O sensor é *signal-off*: com o RNA, o pico cai.

| Seção | O que faz |
|---|---|
| 1. Base | importa os `.nox` de `dados/rna/` para o banco (pico medido, cada RNA ligado ao branco do eletrodo) |
| 2. Curvas | branco × RNA de cada eletrodo |
| 3. Pico e calibração | queda do pico por concentração; reta de calibração e limite de detecção |
| 4. Modelo | 4 representações da curva × 15 algoritmos, validação por eletrodo, teste guardado |
| 5. Previsão | classifica um `.nox` novo com o modelo, como a plataforma faz |

**Como usar:** coloque os arquivos em `dados/rna/contaminado/` (`dpv-RNA-<conc>-Ej.nox`) e
`dados/rna/nao_contaminado/` (`dpv-branco-Ej.nox`) — `fiep rna pastas --entrada dados/rna` cria as
pastas. Padrão de nome e campos: `docs/DICIONARIO_DADOS.md`. Nada de `dados/` vai para o Git.

In [1]:
import os, sys
from pathlib import Path

# o notebook fica em notebook/: trabalhar a partir da raiz do repositório
RAIZ = Path.cwd() if (Path.cwd() / "fiep").exists() else Path.cwd().parent
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from fiep import __version__, db, genossensor as g, modelagem as m

pd.set_option("display.max_columns", 30)
RNA = Path("dados/rna")
BANCO = "dados/fiep.db"
MODELOS = "modelos"
RAPIDO = False        # True: só 3 algoritmos (teste rápido)
print(f"fiep {__version__}")

fiep 0.1.0


## 1. Base

Cada `.nox` vira uma linha da tabela `ensaios`: tipo (branco/RNA), concentração (ng/mL) e eletrodo vêm
do nome do arquivo; o elemento de captura, do procedimento gravado pelo NOVA. O pico é medido na hora.
Arquivo na pasta errada, de outra etapa (`bare`, `tratado`...) ou sem o eletrodo no nome não entra — a
importação diz por quê. Importar de novo não duplica (SHA-256).

In [2]:
con = db.conectar(BANCO)
if any(RNA.rglob("*.nox")):
    imp = g.importar(con, RNA)
    print(f"{imp['lidos']} lido(s), {imp['gravados']} novo(s), {imp['ja_no_banco']} já no banco, "
          f"{imp['nao_gravados']} não gravado(s)")
    for aviso in imp["avisos"]:
        print("  aviso:", aviso)
base = g.tabela(con, incluir_curvas=True)
if base.empty:
    print(f"Sem ensaios no banco. Crie as pastas com `fiep rna pastas --entrada {RNA}` e coloque os .nox nelas.")
else:
    display(base[["arquivo", "eletrodo_id", "tipo", "concentracao", "elemento_captura", "e_pico_v", "i_pico_ua",
                  "altura_ua", "queda_pct", "abaixo_do_branco", "qc_flag"]].round(3))
    print(base.groupby("tipo")["eletrodo_id"].agg(ensaios="size", eletrodos="nunique"))

Sem ensaios no banco. Crie as pastas com `fiep rna pastas --entrada dados/rna` e coloque os .nox nelas.


## 2. Curvas: branco × RNA de cada eletrodo

Linha cheia: o branco; tracejada: com o RNA (mais escura quanto maior a concentração).

In [3]:
if not base.empty:
    eletrodos = sorted(base.eletrodo_id.unique())
    concs = sorted(c for c in base.concentracao.unique() if c > 0)
    cmap = plt.get_cmap("Oranges")
    cor = lambda c: "0.25" if c == 0 else cmap(0.35 + 0.65 * (concs.index(c) / max(len(concs) - 1, 1)))
    fig, axes = plt.subplots(1, len(eletrodos), figsize=(4.2 * len(eletrodos), 3.4), squeeze=False)
    for ax, el in zip(axes[0], eletrodos):
        for r in base[base.eletrodo_id == el].sort_values("concentracao").itertuples():
            ax.plot(r.potential_v, r.current_ua, color=cor(r.concentracao), lw=1.8 if r.tipo == "branco" else 1.2,
                    ls="-" if r.tipo == "branco" else "--", label="branco" if r.tipo == "branco" else f"{r.concentracao:g} ng/mL")
        ax.set_title(f"Eletrodo {el}"); ax.set_xlabel("Potencial (V)"); ax.legend(fontsize=7)
    axes[0][0].set_ylabel("δI (µA)")
    plt.tight_layout(); plt.show()

## 3. Pico e calibração

Queda do pico em relação ao branco do mesmo eletrodo, por concentração. A reta da queda contra
log₁₀(concentração) é a curva de calibração; o limite de detecção (3 × desvio dos brancos ÷ inclinação)
aparece com pelo menos 3 brancos.

In [4]:
if not base.empty:
    rna = base[(base.tipo == "rna") & base.queda_pct.notna()]
    if len(rna):
        fig, ax = plt.subplots(figsize=(5, 3.4))
        ax.scatter(rna.concentracao, rna.queda_pct, color="#c0501f")
        ax.axhline(0, color="0.6", lw=0.8)
        ax.set_xscale("log"); ax.set_xlabel("RNA (ng/mL)"); ax.set_ylabel("queda do pico vs branco (%)")
        plt.tight_layout(); plt.show()
    print("Calibração:", g.calibracao(base))
    contra = base[base.abaixo_do_branco == False]  # noqa: E712
    if len(contra):
        print("RNA com pico ACIMA do branco do eletrodo (o contrário do esperado):", ", ".join(contra.arquivo))

## 4. Modelo pela curva

`fiep.modelagem.treinar` compara 4 representações da curva (inteira, forma, derivada, medidas do pico)
× 15 algoritmos. A validação cruzada nunca separa as medições de um eletrodo entre treino e avaliação;
com 8 ou mais eletrodos, cerca de 25 % deles ficam guardados como teste e o vencedor é medido neles uma
única vez. Precisa de pelo menos 3 brancos e 3 RNA aprovados.

In [5]:
pacote = None
try:
    pacote = m.treinar(base, rapido=RAPIDO)
except ValueError as e:
    print("Ainda não dá para treinar:", e)
if pacote:
    c = pacote["cartao"]
    print(f"Escolhido: {c['algoritmo_nome']} · {c['representacao_nome']}")
    print(f"Acurácia balanceada: {c['ab_referencia']:.3f} ({c['referencia']}) — {c['validacao']}")
    for a in c["avisos"]:
        print("AVISO:", a)
    placar = pd.DataFrame(c["placar"])
    placar["algoritmo"] = placar["algoritmo"].map(m.nome_classificador)
    placar["representacao"] = placar["representacao"].map(lambda v: m.INFO_VARIANTES[v]["nome"])
    display(placar.head(15)[["posicao", "representacao", "algoritmo", "balanced_accuracy", "roc_auc",
                             "sensibilidade", "especificidade"]].round(3))
    # m.salvar(pacote, MODELOS, s=con)   # descomente para registrar e usar na plataforma

Ainda não dá para treinar: a base está vazia: importe os .nox de branco e de RNA primeiro (fiep rna importar)


## 5. Previsão de um arquivo novo

O mesmo que a tela Previsão e o `fiep rna prever`: lê o `.nox`, confere a faixa de potencial e classifica.
Aqui, por demonstração, os próprios arquivos da base.

In [6]:
if pacote is not None:
    arquivos = sorted(RNA.rglob("*.nox"))[:6]
    res = pd.DataFrame(m.prever_arquivos(pacote, arquivos))
    display(res[["arquivo", "tipo_informado", "resultado", "prob_contaminado", "motivo"]])

## Próximos passos

- Mais eletrodos, cada um com o branco e o RNA, e repetições de cada concentração: é o que dá teste
  independente e limite de detecção.
- Controles negativos (amostra sem o RNA-alvo) para separar "tem RNA" de "o eletrodo recebeu amostra".
- Resultados para o artigo quando houver dados suficientes.

In [7]:
con.close()